In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
W, H = 2.9464, 2.26

In [ ]:
def show_trajectory(ax, x, y, ls='r', **line_kwargs):
    # First draw the cable robot
    corners = np.array([
        [0, 0],
        [W, 0],
        [W, H],
        [0, H],
        [0, 0]
    ])
    ax.plot(*corners.T, 'k', lw=2)
    ax.plot(x, y, ls, **line_kwargs)

In [ ]:
# We want to create a zig-zag trajectory
xs = np.array([0.75, 2.25])
ys = np.arange(1.5, 0.49, -0.25)
xs = np.tile(np.concatenate([xs, xs[::-1]]), (len(ys) + 1) // 2)
ys = ys.repeat(2)
xs = xs[:len(ys)]
print(xs)
print(ys)

# Plot
plt.figure(figsize=(4, 3))
show_trajectory(plt.gca(), xs, ys)
plt.axis('equal');

In [ ]:
# Now we want to discretize the trajectory
def discretize(ds, xs, ys):
    xys = []
    for i in range(len(xs) - 1):
        x0, y0 = xs[i], ys[i]
        x1, y1 = xs[i + 1], ys[i + 1]
        l = np.hypot(x1 - x0, y1 - y0)
        n = int(np.ceil(l / ds))
        xys.extend(zip(np.linspace(x0, x1, n), np.linspace(y0, y1, n)))
    return np.array(xys)
xys = discretize(0.1, xs, ys)

# Plot
plt.figure(figsize=(4, 3))
show_trajectory(plt.gca(), xys[:, 0], xys[:, 1], ls='rx-')
plt.axis('equal');

In [ ]:
traj = discretize(0.01, xs, ys)

In [ ]:
# Export as gcode
def export(fname, xys):
    with open(fname, 'w') as f:
        xs, ys = xys.T * 1e3
        # Write path to file
        f.write(f'G0 X{xs[0]:.3f} Y{ys[0]:.3f}\n')
        for x, y in zip(xs, ys):
            f.write(f'G1 X{x:.3f} Y{y:.3f}\n')
        f.write(f'G0 X{xs[-1]:.3f} Y{ys[-1]:.3f}\n')

export('zigzag.nc', traj)
np.save('zigzag.npy', traj)